In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import os
from zhipuai import ZhipuAI
def fire(num):
    page = 31584
    chrome_options = Options()
    chrome_options.page_load_strategy = 'normal'
    driver = webdriver.Chrome(options=chrome_options)
    for i in range(31583,34360):
        print("开始爬取第 %s 页" % page)
        url = 'https://guba.eastmoney.com/list,zssh000001_{}.html'.format(page)
        num=get_poster_url(url,num,driver)
        page += 1
    driver.quit()
def get_poster_url(url,num,driver):
    driver.get(url)
    if num==1059510:
        element1 = driver.find_element(By.XPATH, "//img[contains(@onclick, 'tk_tg_zoomin()')]")
        element1.click()
        element2 = driver.find_element(By.XPATH, "//*[@data-tracker-eventcode='gb.ggb.qhzy']")
        element2.click()
        element3 = driver.find_element(By.XPATH, "//*[@data-tracker-eventcode='gb.ggb.zxft']")
        element3.click()
        time.sleep(0.5)
    time.sleep(0.3)
    element3 = driver.find_element(By.XPATH, "//*[@data-tracker-eventcode='gb.ggb.zxft']")
    element3.click()
    data1 = driver.find_elements(By.XPATH,"//ul[@id='list']/li/div[2]/div[1]/div[2]")
    data2 = driver.find_elements(By.XPATH,"//ul[@id='list']/li/div[2]/a/div[2]")
    for i,j in zip(data1,data2):
        try:
            date=i.text
            date=date.split(" ")[1]
            if date.startswith("0"):
                date="2025-"+date
            elif date.startswith("24"):
                date="20"+date
            if date>"2025-03-24" or date<"2024-03-24":
                continue
            content=j.text
            if content=="":
                continue
            sen=sentiment(content)
            num+=1
            p_name = '帖子{}.txt'.format(num)
            l_name = '{}\\'.format(date)
            with open('帖子\\'+l_name+ p_name, 'w',encoding='utf8') as f:
                f.write(date+'\n')
                f.write(sen+'\n')
                f.write(content+'\n')
        except:
            print(url)
            print(date)
            print(content)
    return num
def sentiment(text):
    client = ZhipuAI(api_key="YOUR_ZHIPUAI_API_KEY")  # 请填写您自己的APIKey
    response = client.chat.completions.create(
    model="glm-4-flash",  # 请填写您要调用的模型名称
    messages=[
        {"role": "system", "content": "你是一个金融学专家，请给出下面股吧帖子的情绪，若它是积极的则输出“1”，消极的输出“-1”，中立的输出“0”，只能输出以上三个数字"},
        {"role": "user", "content": text}
             ],
         )
    sen = response.choices[0].message.content
    return sen       
    
num=1059510
fire(num)

In [ ]:
import os
fileNameList = [item for item in os.listdir('帖子/情感词典/')]
num=0
for fileName in fileNameList:
    try:
        file = open('帖子/情感词典/' + fileName, encoding='utf8')
        date=file.readline()
        content1=file.read()
        date=date.strip()
        content1=content1.strip()
        content=content1.split("#")[0]
        if content=="":
            continue
        sen=sentiment(content)
        num+=1
        l_name = '{}\\'.format(date)
        with open('帖子\\'+l_name+ fileName, 'w',encoding='utf8') as f:
            f.write(date+'\n')
            f.write(sen+'\n')
            f.write(content+'\n')
        print(num)
    except:
        print(fileName)


In [ ]:
import os
import math
fileNameList = [item for item in os.listdir('帖子_2/')]
for fileName in fileNameList:
    filelist = [item for item in os.listdir('帖子_2/'+fileName+'/')]
    l=len(filelist)
    l=str(l)
    p=0
    n=0
    for i in filelist:
        file = open('帖子_2/' + fileName+"/"+i, encoding='utf8')
        file.readline()
        sen=file.readline()
        sen=sen.strip()
        if sen=='-1':
            n=n+1
        elif sen=='1':
            p=p+1
    s=math.log((1+p)/(1+n))
    s=str(s)
    with open('帖子_2\\'+"信息量.txt", 'a',encoding='utf8') as f:
        f.write(fileName+"——"+l+'\n')
    with open('帖子_2\\'+"情绪指数.txt", 'a',encoding='utf8') as k:
        k.write(fileName+"——"+s+'\n')


In [ ]:
file = open('帖子_2/情绪指数.txt', encoding='utf8')
File = open('帖子_2/信息量.txt', encoding='utf8')
for i in range(0,183):
    a=file.readline()
    a=a.strip().split("——")[1]
    b=File.readline()
    b=b.strip().split("——")[1]
    with open('帖子_1\\'+"情绪指数.xlsx", 'a',encoding='utf8') as f:
        f.write(a+'\n')
    with open('帖子_1\\'+"信息量.xlsx", 'a',encoding='utf8') as k:
        k.write(b+'\n')

In [ ]:
import os
from re import sub
from jieba import cut
from collections import Counter
from itertools import chain
fileNameList = [item for item in os.listdir('帖子/')]
fileNameList_1=fileNameList[184:204]
words=[]
for fileName in fileNameList_1:
    filelist = [item for item in os.listdir('帖子/'+fileName+'/')]
    for i in filelist:
        file = open('帖子/' + fileName+"/"+i, encoding='utf8')
        file.readline()
        file.readline()
        text=file.read()
        text=text.strip()
        text=sub(r'[.【】0-9、—。，！~\*]', '', text)
        text = cut(text)
        text = filter(lambda j: len(j)>1, text)
        words.append(text)
freq = Counter(chain(*words))
a=[w[0] for w in freq.most_common(100)]
print(a)

In [ ]:
print(freq)